## TUGAS MANDIRI - Debi Aprilia - 2320506038
### Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [2]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


**Sebelum mengerjakan tugas mandiri poin A - E buat SparkSession terlebih dahulu.**

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count

#Membuat SparkSession
spark = SparkSession.builder \
    .appName("TugasMandiriPertemuan4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat!")

SparkSession berhasil dibuat!


**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).


In [6]:
#Membaca data dari HDFS
file_path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df_tugas = spark.read.csv(file_path_hdfs, header=True, inferSchema=True)

#Menampilkan Schema
df_tugas.printSchema()

#Menampilkan jumlah baris
print("Jumlah total baris: ", df_tugas.count())

#Menampilkan 10 baris pertama
df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah total baris:  1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang| 

**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [7]:
#Menampilkan atau menghitung jumlah data kosong pada kolom rating
jumlah_null = df_tugas.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating NULL: {jumlah_null}")

#Mengisi nilai kosong
df_clean = df_tugas.fillna({"rating": 0})

#Untuk verifikasi
print("Jumlah NULL setelah penanganan:", df_clean.filter(col("rating").isNull()).count())

Jumlah baris dengan rating NULL: 204
Jumlah NULL setelah penanganan: 0


**Alasan memilih df.na.fill() atau df.fillna()** yaitu mengisi nilai NULL dengan 0 adalah untuk mempertahankan jumlah data agar transaksi tidak terhapus dan data tetap utuh untuk analisis berikutnya. Selain itu, dengan mengganti nilai NULL dengan 0 akan membuat baris tetap bisa diolah dalam operasi matematika atau agregasi selanjutnya tanpa menghasilkan eror.

**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya

In [8]:
#Membuat kolom baru yaitu total_pendapatan dan tier_transaksi
df_transformed = df_clean \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

#Menampilkan 10 baris pertama pada kolom baru yang berhasil dibuat
df_transformed.select("order_id", "total_pendapatan", "tier_transaksi").show(10)

[Stage 11:>                                                         (0 + 1) / 1]

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
|ORD-3005|          100000|         Kecil|
|ORD-3006|           40000|         Kecil|
|ORD-3007|          720000|         Besar|
|ORD-3008|          140000|         Kecil|
|ORD-3009|          900000|         Besar|
+--------+----------------+--------------+
only showing top 10 rows



**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?

In [9]:
from pyspark.sql.functions import sum as spark_sum

df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show()

[Stage 12:>                                                         (0 + 1) / 1]

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



Kategori yang memiliki total pendapatan tertinggi adalah kategori Rumah Tangga dengan total pendapatan sebesar Rp 138.665.000.

2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?

In [11]:
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

[Stage 15:>                                                         (0 + 1) / 1]

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



Kota yang memiliki jumlah transaksi tier "Besar" terbanyak adalah kota Solo dengan jumlah 92.

3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

In [12]:
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .show()

[Stage 18:>                                                         (0 + 1) / 1]

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

In [13]:
output_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transformasi"
df_transformed.write.mode("overwrite").csv(output_hdfs, header=True)

print("Data berhasil disimpan ke HDFS.")

Data berhasil disimpan ke HDFS.


In [14]:
#Untuk verifikasi apakah sudah berhasil
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transformasi

Found 2 items
-rw-r--r--   3 deb supergroup          0 2026-09-16 20:24 /user/mahasiswa/tugas4/hasil_transformasi/_SUCCESS
-rw-r--r--   3 deb supergroup      97296 2026-09-16 20:24 /user/mahasiswa/tugas4/hasil_transformasi/part-00000-f91694bb-38e8-4825-82f5-1cc408280a0f-c000.csv


> **Catatan:** Spark menyimpan hasil sebagai **beberapa berkas partisi** (`part-00000...`, dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark. Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi

Spark menyimpan hasil sebagai beberapa berkas partisi karena Spark memang didesain untuk komputasi terdistribusi *(distributed computing)*. Saat proses penulisan data, setiap *worker/thread* mengeksekusi sebagian data (partisi) secara independen dan paralel, lalu hasilnya ditulis ke dalam filenya masing - masing. Hal ini membuat proses eksekusi data berskala besar menjadi lebih efisien daripada menggabungkan seluruh data menjadi satu file tunggal yang dapat menyebabkan *bottleneck* memori.

**Menutup SparkSession ketika telah selesai digunakan.**

In [15]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
